In [ ]:
# | default_exp content.analysis

# Content Analysis
> DB-access functions for heavy computation: keyword density and cannibalization.

In [ ]:
# | export
def calculate_keyword_density(content: str,
                              keyword: str
                              ) -> dict:
    "Calculate keyword density using word-level matching."
    try:
        content_words = content.lower().split()
        keyword_words = keyword.lower().split()
        n = len(keyword_words)
        positions = [i for i in range(len(content_words) - n + 1)
                     if content_words[i:i + n] == keyword_words]
        total = len(content_words)
        return {"keyword": keyword, "count": len(positions),
                "density": (len(positions) / total * 100) if total > 0 else 0,
                "positions": positions}
    except Exception:
        return {"keyword": keyword, "count": 0, "density": 0.0, "positions": []}


In [ ]:
#| export
from seootter.article import get_articles_by_website
from seootter.parser.utils import get_page_content
from seootter.sqlite_db import get_session
from seootter.models import GSCAnalytics
from seootter.gsc.queries import get_top_queries
from seootter.gsc_client import get_date_range
from sqlmodel import select


In [ ]:
# | export
def find_cannibalized(session,
                      website_id: int,
                      site_url: str,
                      start: str,
                      end: str
                      ) -> dict:
    "Find keyword cannibalization via exact focus keyword matches and GSC ranking data."
    try:
        from collections import defaultdict
        articles = get_articles_by_website(session=session, website_id=website_id)
        kw_map = defaultdict(list)
        for a in articles:
            if a.focus_keyword: kw_map[a.focus_keyword].append(a.url)
        exact_matches = [{"keyword": kw, "pages": urls}
                         for kw, urls in kw_map.items() if len(urls) >= 2]
        gsc_matches = []
        for query in [r["query"] for r in get_top_queries(session, site_url, start, end, limit=1000)]:
            pages = session.exec(
                select(GSCAnalytics.page).where(
                    GSCAnalytics.site_url == site_url, GSCAnalytics.query == query,
                    GSCAnalytics.date.between(start, end), GSCAnalytics.position <= 20,
                ).distinct()
            ).all()
            if len(pages) >= 2: gsc_matches.append({"query": query, "pages": pages})
        return {"exact_matches": exact_matches, "gsc_matches": gsc_matches}
    except Exception:
        return {"exact_matches": [], "gsc_matches": []}